In [1]:
#import part
import src.Astar_route as astar
import random
import src.make_map_list as mml
from src.Map_visualize import map_visualize as mapv
import src.make_order as mo
from src.visualize_simulation import visualize_simulation as vs
import csv
import numpy as np
import pandas as pd
import time
import os
from IPython.display import clear_output
from src.agent_class import Agent
from src.map_class import Map
from src.order_class import Order

In [2]:
#config part

num_node = 30 #ノード数
gen_prob = 0.3
deadtime = 100
current_time = 0
order_id_counter = 1
num_agent= 3
attitude = astar.a_star
agents= {}
vehicle_speed =75

ST_PENDING = "PENDING"
ST_ASSIGNED = "ASSIGNED"
ST_COMPLETED = "COMPLETED"

log_filename = "simulation_log_003.csv"

In [3]:
# --- 事前準備（3つ目のセルでやってある想定） ---
# 設計図(Map)から、実体(sim_map)を作っておく
sim_map = Map(num_node=30)

# --- エージェントの生成（4つ目のセル） ---
agents = {}

for i in range(num_agent):
    # 💡 クラスの復習！
    # sim_map という実体のポケットから、文字列のノードリスト（['0', '1', ...])を直接引っ張ってくる！
    node_list = sim_map.node_list
    
    # リストからランダムに1つ選択（これでエラーが消えます）
    start_node = random.choice(node_list)
    
    # 前に「if文を入れて被らないようにしたい」と言っていたロジックをここで発動する場合：
    # existing_nodes = [ag.current_node for ag in agents.values()]
    # while start_node in existing_nodes:
    #     start_node = random.choice(node_list)

    agent_id = f"AMR_{i:03d}"  # IDを文字列にしておくとログが見やすくなります

    # Agentクラスのキーワード引数に合わせて生成
    agents[i] = Agent(
        attitude=attitude,
        agent_id=agent_id,
        start_node=start_node,
        vehicle_speed=vehicle_speed
    )

print(f"--- エージェントが {len(agents)} 台生成されました ---")
for idx, ag in agents.items():
    print(f"Key: {idx} | ID: {ag.agent_id} | 初期位置: ノード {ag.current_node}")

--- エージェントが 3 台生成されました ---
Key: 0 | ID: AMR_000 | 初期位置: ノード 25
Key: 1 | ID: AMR_001 | 初期位置: ノード 13
Key: 2 | ID: AMR_002 | 初期位置: ノード 4


In [4]:
import math
import os
import time
import numpy as np
import pandas as pd




# 直線距離を計算するヘルパー関数（sim_mapの座標ポケットから位置を引くように改良！）
def get_straight_distance(node_a, node_b, sim_map):
    pos_a = sim_map.get_pos(node_a)
    pos_b = sim_map.get_pos(node_b)
    return math.hypot(pos_a[0] - pos_b[0], pos_a[1] - pos_b[1])


# アクティブオーダーの管理（中身は Order クラスの実体が入る）
active_orders = {}
current_time = 0
order_id_counter = 1

print("クラス連動シミュレーションを開始します... (Ctrl+C で停止)")

try:
    while True:

        # --- STEP 1: マップ環境のアップデート ---
        # これを叩くだけで、Mapクラスの内部で全道路の渋滞がじわじわ変化します
        sim_map.update()

        # --- STEP 2: 新しいオーダーの発生（Orderクラスのインスタンス化） ---
        if np.random.rand() < gen_prob:
            oid = f"ORD_{order_id_counter:05d}"

            # 関数ではなく、Order クラスの設計図から実体（インスタンス）を生成！
            new_ord = Order(
                order_id=oid,
                current_time=current_time,
                deadtime=deadtime,
                num_node=num_node,
            )
            active_orders[oid] = new_ord
            order_id_counter += 1

        # --- STEP 3: 配分マッチング（管制塔のロジック） ---
        for oid, order in active_orders.items():
            if order.status == "PENDING":  # 💡クラスなので辞書型ではなくドット記法

                best_agent = None
                min_distance = float("inf")

                # 待機中（IDLE）のロボットの中で、荷物位置（origin）に一番近い子を探す
                for ag in agents.values():
                    if ag.status == "IDLE":
                        # 改良した直線距離計算に関数を引き渡す
                        dist = get_straight_distance(
                            ag.current_node, order.origin, sim_map
                        )
                        if dist < min_distance:
                            min_distance = dist
                            best_agent = ag

                # 一番近いロボットが見つかったら、仕事を正式に割り当てる
                if best_agent is not None:
                    # エージェントにマップクラス（sim_map）ごと仕事を渡す
                    best_agent.assign_order(order, sim_map)
                    order.status = "ASSIGNED"  # 💡ドット記法でステータス更新
                    print(
                        f"🤖 [配分] {best_agent.agent_id} が オーダー {oid} を担当します"
                    )

        # --- STEP 4: エージェントの移動 ＆ 経路リプランニング ---
        for ag in agents.values():
            # エージェントにマップクラスを渡して1秒進める
            # この内部で「渋滞を考慮した移動時間の計算」や「交差点に着くたびのA*再計算」が自動で走ります
            ag.update(sim_map)

        # --- STEP 5: 完了したオーダーの回収（CSV保存 ＆ メモリ解放） ---
        completed_ids = [
            oid
            for oid, order in active_orders.items()
            if order.status == "COMPLETED"  # 💡ドット記法
        ]

        if completed_ids:
            completed_list = []
            for oid in completed_ids:
                order_obj = active_orders.pop(oid)  # クラスの実体を辞書から完全削除

                # 💡 DataFrameに渡すために、クラス内で作った .to_dict() で辞書型に戻す！
                completed_list.append(order_obj.to_dict())

            # CSVに追記保存
            df_to_save = pd.DataFrame(completed_list)
            df_to_save.to_csv(
                log_filename,
                mode="a",
                header=not os.path.exists(log_filename),
                index=False,
            )

        # --- STEP 6: 定期的なコンソール表示 (10ステップごと) ---
        if current_time % 10 == 0:
            print(
                f"\n⏱️ Time: {current_time}s | 稼働中オーダー: {len(active_orders)}件"
            )
            for ag in agents.values():
                loc = (
                    ag.current_node
                    if ag.next_node is None
                    else f"{ag.current_node}➔{ag.next_node}"
                )
                print(f"  ┗ {ag.agent_id} [位置: {loc:<7} | 状態: {ag.status}]")

        clear_output(wait=True)

        # ② 新しい状態のマップとロボットたちをミリ秒単位の進捗で綺麗に描画！
        vs(sim_map, agents, active_orders, current_time)

    current_time += 1
    time.sleep(0.2)  # アニメーションのコマ送り速度調整

except KeyboardInterrupt:
    print("\nシミュレーションを終了しました。")



AttributeError: 'Map' object has no attribute 'items'

<Figure size 1200x1000 with 0 Axes>